# Part 4: Performance Arena

Chunking comparison, CAG simulation, CRAG impact and cost analysis.

In [1]:
from prime_land.chat_service.cag_cache import CAGCache
from prime_land.chat_service.cag_service import CAGService
from prime_land.chat_service.crag_service import CRAGService
from prime_land.chat_service.rag_service import RAGService
from prime_land.config import CONFIG, get_path, load_faqs
from prime_land.evaluation.cag_eval import simulate_cag
from prime_land.evaluation.chunking_eval import evaluate_chunking
from prime_land.evaluation.cost_analysis import analyze_cost
from prime_land.evaluation.crag_eval import evaluate_crag
from prime_land.ingest_document.indexer import collection_name, get_client, open_store
from prime_land.providers import get_embeddings, get_llm

embeddings = get_embeddings()
llm = get_llm()
client = get_client()
stores = {strategy: open_store(client, embeddings, strategy) for strategy in CONFIG["chunking"]["strategies"]}

## Chunking strategy comparison

In [2]:
chunking_table = evaluate_chunking(stores, embeddings, llm)
chunking_table

evaluating 1/50 [semantic]: Which gated house community is available in Kalutara?
evaluating 2/50 [semantic]: What is the price of land at Solara Meepe?
evaluating 3/50 [semantic]: Tell me about Veloria in Bokundara
evaluating 4/50 [semantic]: Which houses are available in Thalawathugoda?
evaluating 5/50 [semantic]: What amenities does Kaloora offer?
evaluating 6/50 [semantic]: Are there lands near Galle?
evaluating 7/50 [semantic]: What is the payment plan for Kaloora?
evaluating 8/50 [semantic]: Where is Scottish Island located?
evaluating 9/50 [semantic]: Which land is available in Kadawatha?
evaluating 10/50 [semantic]: How can I contact Prime Lands?
evaluating 11/50 [fixed]: Which gated house community is available in Kalutara?
evaluating 12/50 [fixed]: What is the price of land at Solara Meepe?
evaluating 13/50 [fixed]: Tell me about Veloria in Bokundara
evaluating 14/50 [fixed]: Which houses are available in Thalawathugoda?
evaluating 15/50 [fixed]: What amenities does Kaloora o

,strategy,precision_at_5,recall_at_5,answer_relevance,latency_s,score
2,parent_child,0.96,1.0,0.788,22.946,2.748
3,semantic,0.92,1.0,0.791,23.256,2.711
4,sliding,0.92,1.0,0.777,22.955,2.697
0,fixed,0.80,1.0,0.792,23.043,2.592
1,late,0.76,1.0,0.783,22.995,2.543


## CAG cache effectiveness

In [3]:
strategy = CONFIG["retrieval"]["strategy"]
rag = RAGService(stores[strategy], embeddings, llm, strategy=strategy)
cache = CAGCache(embeddings, get_path("cache_dir"), CONFIG["cag"]["similarity_threshold"], CONFIG["cag"]["history_ttl_hours"])
cag = CAGService(rag, cache)
cag.warm_faqs(load_faqs())
cag_stats = simulate_cag(cag, load_faqs())
cag_stats

simulating 1/100 hit 0.5s
simulating 2/100 hit 0.0s
simulating 3/100 hit 0.0s
simulating 4/100 hit 0.4s
simulating 5/100 hit 0.4s
simulating 6/100 hit 0.4s
simulating 7/100 miss 21.0s
simulating 8/100 hit 0.0s
simulating 9/100 hit 0.5s
simulating 10/100 hit 0.0s
simulating 11/100 hit 0.5s
simulating 12/100 hit 0.0s
simulating 13/100 hit 0.0s
simulating 14/100 hit 0.4s
simulating 15/100 hit 0.0s
simulating 16/100 hit 0.0s
simulating 17/100 hit 0.0s
simulating 18/100 hit 0.0s
simulating 19/100 hit 0.4s
simulating 20/100 hit 0.4s
simulating 21/100 hit 0.0s
simulating 22/100 hit 0.0s
simulating 23/100 hit 0.0s
simulating 24/100 hit 0.4s
simulating 25/100 miss 21.3s
simulating 26/100 hit 0.5s
simulating 27/100 hit 0.0s
simulating 28/100 hit 0.0s
simulating 29/100 miss 24.4s
simulating 30/100 hit 0.5s
simulating 31/100 hit 0.0s
simulating 32/100 hit 0.4s
simulating 33/100 hit 0.0s
simulating 34/100 miss 22.2s
simulating 35/100 hit 0.0s
simulating 36/100 hit 0.0s
simulating 37/100 hit 0.0s
si

{'total_queries': 100,
 'hits': 94,
 'misses': 6,
 'faq_hits': 88,
 'history_hits': 6,
 'hit_rate': 0.94,
 'avg_hit_latency_s': 0.239,
 'avg_miss_latency_s': 22.04,
 'latency_improvement_pct': 98.9,
 'cost_per_uncached_query_usd': 9.9e-05,
 'estimated_savings_usd': 0.009306,
 'api_calls_avoided': 94}

## CRAG correction impact

In [4]:
crag = CRAGService(rag, llm)
crag_table, crag_summary = evaluate_crag(rag, crag, llm)
print(crag_summary)
crag_table

comparing 1/20: Which gated house community is available in Kalutara?
comparing 2/20: What is the price of land at Solara Meepe?
comparing 3/20: Tell me about Veloria in Bokundara
comparing 4/20: Which houses are available in Thalawathugoda?
comparing 5/20: What amenities does Kaloora offer?
comparing 6/20: Are there lands near Galle?
comparing 7/20: What is the payment plan for Kaloora?
comparing 8/20: Where is Scottish Island located?
comparing 9/20: Which land is available in Kadawatha?
comparing 10/20: How can I contact Prime Lands?
comparing 11/20: Which property is best for a family wanting security and a pool?
comparing 12/20: What is the cheapest house I can buy?
comparing 13/20: Can I pay in installments without interest?
comparing 14/20: Which projects are close to the highway?
comparing 15/20: I want a quiet land by the beach in the south
comparing 16/20: Do you offer bank loan facilities?
comparing 17/20: Which apartments are available?
comparing 18/20: Show me upcoming pro

,question,rag_confidence,crag_initial_confidence,crag_final_confidence,correction_applied,rag_quality,crag_quality,confidence_gain,quality_gain
0,Which gated house community is available in Ka...,0.798,0.798,0.798,False,5,5,0.000,0
1,What is the price of land at Solara Meepe?,0.759,0.759,0.759,False,5,5,0.000,0
2,Tell me about Veloria in Bokundara,0.849,0.849,0.849,False,5,5,0.000,0
3,Which houses are available in Thalawathugoda?,0.667,0.667,0.667,False,5,5,0.000,0
4,What amenities does Kaloora offer?,0.639,0.639,0.639,False,5,5,0.000,0
5,Are there lands near Galle?,0.794,0.794,0.794,False,4,4,0.000,0
6,What is the payment plan for Kaloora?,0.742,0.742,0.742,False,5,5,0.000,0
7,Where is Scottish Island located?,0.505,0.505,0.699,True,5,5,0.194,0
8,Which land is available in Kadawatha?,0.918,0.918,0.918,False,3,5,0.000,2
9,How can I contact Prime Lands?,0.955,0.955,0.955,False,5,5,0.000,0


## Cost analysis (bonus)

In [5]:
vector_count = client.count(collection_name(strategy)).count
analyze_cost(cag_stats, vector_count)

{'monthly_queries': 75000,
 'cache_hit_rate': 0.94,
 'api_cost_without_cache_usd': 7.76,
 'api_cost_with_cache_usd': 0.78,
 'storage_cost_usd': 0.0004,
 'monthly_system_cost_usd': 0.78,
 'monthly_cache_savings_usd': 6.98,
 'human_agent_cost_usd': 37500.0,
 'roi_percent': 4786449.0}

In [6]:
client.close()